# arXiv:2208.03923 — Core Pullback-Metric Reproduction

This notebook tests the paper's core local-geometry mechanism on our existing trained ConvVAE before we move to TNAR/ManiFool.

We compute $\hat G_x=J_\mu(x)^T J_\mu(x)+J_\sigma(x)^T J_\sigma(x)$, eigendecompose it, perturb along its dominant eigendirection, and compare that direction against random directions at matched input-space $L_2$ norm.

This is **Phase A: method reproduction on our model**, not yet the full paper-specific beta-VAE architecture sweep.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms

ROOT = Path.cwd()
if ROOT.name == 'reproductions':
    ROOT = ROOT.parent.parent
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.vae import ConvVAE
from src.latent_geometry.pullback_metric import (
    direction_sweep, latent_mean_shift, matched_random_direction,
    mean_reconstruction, paper_anisotropy_score,
    perturb_along_eigendirection, stochastic_pullback_metric,
)


## 1. Load our trained VAE and one MNIST sample

Point `CHECKPOINT_PATH` to the VAE checkpoint from the existing project. Checkpoints are intentionally not committed to Git.


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LATENT_DIM = 16
CHECKPOINT_PATH = ROOT / 'checkpoints' / 'vae_model.pth'

vae = ConvVAE(latent_dim=LATENT_DIM).to(DEVICE)
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}. Update CHECKPOINT_PATH.')

state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
vae.load_state_dict(state)
vae.eval()

test_set = datasets.MNIST(ROOT / 'data', train=False, download=True, transform=transforms.ToTensor())
INDEX = 0
x, y = test_set[INDEX]
x = x.unsqueeze(0).to(DEVICE)
print('device:', DEVICE, 'label:', y, 'shape:', tuple(x.shape))


## 2. Compute the stochastic pullback metric and check invariants

For MNIST the metric is $784\times784$. It should be symmetric and positive semidefinite up to numerical precision.


In [ ]:
geometry = stochastic_pullback_metric(vae, x)
G = geometry.metric
eigvals = geometry.eigenvalues
eigvecs = geometry.eigenvectors

print('G shape:', tuple(G.shape))
print('J_mu:', tuple(geometry.jacobian_mu.shape), 'J_sigma:', tuple(geometry.jacobian_sigma.shape))
print('symmetry error:', torch.linalg.matrix_norm(G - G.T).item())
print('minimum eigenvalue:', eigvals.min().item())
print('spectral radius:', geometry.spectral_radius.item())
print('paper anisotropy score:', paper_anisotropy_score(eigvals).item())

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(eigvals[:50].detach().cpu().numpy(), marker='o', markersize=3)
ax.set(title='Top 50 pullback eigenvalues', xlabel='rank', ylabel='eigenvalue')
ax.grid(alpha=0.2)
plt.show()

dominant = eigvecs[:,0].reshape(28,28).detach().cpu()
fig, axes = plt.subplots(1,3,figsize=(10,3))
axes[0].imshow(x[0,0].detach().cpu(), cmap='gray'); axes[0].set_title('original')
axes[1].imshow(mean_reconstruction(vae,x)[0,0].detach().cpu(), cmap='gray'); axes[1].set_title('clean recon')
axes[2].imshow(dominant, cmap='coolwarm'); axes[2].set_title('dominant direction')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()


## 3. Paper-style one-step sweep

The public author code uses `np.logspace(-1, 1, 40)`, i.e. 0.1 to 10. The paper prose elsewhere mentions a lower endpoint of 0.01; we follow the public code here and document the discrepancy.


In [ ]:
DELTAS = np.logspace(-1, 1, 40)
df = pd.DataFrame(direction_sweep(vae, x, geometry, DELTAS, direction_index=0, scale_by_eigenvalue=True))
display(df.head())

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogx(df['delta'], df['recon_mse_to_original'], label='reconstruction MSE')
ax.axhline(df['clean_recon_mse'].iloc[0], linestyle='--', label='clean reconstruction MSE')
ax.set(xlabel='delta', ylabel='MSE to original', title='Reconstruction degradation along dominant direction')
ax.legend(); ax.grid(alpha=0.2); plt.show()

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogx(df['delta'], df['latent_shift'])
ax.set(xlabel='delta', ylabel='latent mean shift', title='Latent displacement along dominant direction')
ax.grid(alpha=0.2); plt.show()

SHOW_DELTAS = [0.5233, 0.7443]
fig, axes = plt.subplots(len(SHOW_DELTAS), 3, figsize=(9,6))
for row, delta in enumerate(SHOW_DELTAS):
    x_corrupt = perturb_along_eigendirection(x, eigvals[0], eigvecs[:,0], delta)
    recon = mean_reconstruction(vae, x_corrupt)
    axes[row,0].imshow(x_corrupt[0,0].detach().cpu(), cmap='gray'); axes[row,0].set_title(f'corrupt d={delta}')
    axes[row,1].imshow(recon[0,0].detach().cpu(), cmap='gray'); axes[row,1].set_title('reconstruction')
    axes[row,2].imshow((x_corrupt-x)[0,0].detach().cpu(), cmap='coolwarm'); axes[row,2].set_title('difference')
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()


## 4. Critical control: dominant direction vs random directions

A single dramatic reconstruction is not enough. Here we disable eigenvalue scaling and match the input-space $L_2$ step so that we test the **direction itself**.


In [ ]:
rng = torch.Generator(device=eigvecs.device); rng.manual_seed(42)
N_RANDOM = 25
EPSILONS = np.linspace(0.01, 1.0, 25)
records = []
eta_top = eigvecs[:,0]

for eps in EPSILONS:
    top_adv = perturb_along_eigendirection(x, 1.0, eta_top, float(eps), scale_by_eigenvalue=False)
    top_recon = mean_reconstruction(vae, top_adv)
    records.append({'epsilon':eps, 'kind':'top_eigenvector', 'recon_mse':F.mse_loss(top_recon,x).item(), 'latent_shift':latent_mean_shift(vae,x,top_adv).item()})

    random_mse, random_latent = [], []
    for _ in range(N_RANDOM):
        direction = matched_random_direction(eta_top, generator=rng)
        adv = perturb_along_eigendirection(x, 1.0, direction, float(eps), scale_by_eigenvalue=False)
        recon = mean_reconstruction(vae, adv)
        random_mse.append(F.mse_loss(recon,x).item())
        random_latent.append(latent_mean_shift(vae,x,adv).item())
    records.append({'epsilon':eps, 'kind':'random_mean', 'recon_mse':float(np.mean(random_mse)), 'latent_shift':float(np.mean(random_latent))})

control = pd.DataFrame(records)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for kind, group in control.groupby('kind'):
    axes[0].plot(group['epsilon'], group['latent_shift'], label=kind)
    axes[1].plot(group['epsilon'], group['recon_mse'], label=kind)
axes[0].set(title='Latent shift: top vs random', xlabel='matched input L2 step', ylabel='latent shift')
axes[1].set(title='Reconstruction effect: top vs random', xlabel='matched input L2 step', ylabel='MSE to original')
for ax in axes: ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


# Verdict

Treat Phase A as positive only if this behavior persists over multiple test images: (1) the metric is numerically symmetric/PSD, (2) its spectrum is non-trivial, (3) dominant-direction latent displacement grows with the step, (4) reconstruction degradation grows with the step, and (5) the dominant direction is consistently more sensitive than matched random directions.

If this works, the next reproduction is the paper-specific 32-D MLP beta-VAE sweep. If it fails on our ConvVAE, do not conclude that the paper is wrong; first separate architecture/training differences from implementation errors.
